In [20]:

from app.services.retriever import search_children 

query = "درباره agentic rag توضیح بده و درباره پایگاه داده برداری هم توضیح بده"

child_results = search_children(query,collection_name="loader_children" ,top_k=10)

In [21]:
child_results

[ChildSearchResult(parent_title='پیاده سازی عملی با LangChain', child_title='پیاده سازی عملی با LangChain', child_content='LangChain یکی از پرکاربردترین چارچوب ها برای ساخت سیستم های RAG   است. در ادامه یک نمونه\nساده از پیاده سازی این معماری ارائه می شود.\nمرحله۱: بارگذاری اسناد\nدر این مرحله از کالس های loader   مانندPyPDFLoader   یاDirectoryLoader   برای خواندن اسناد\nاستفاده می شود.\nمرحله۲: تقسیم بندی متن\nپس از بارگذاری، متن ها با استفاده ازRecursiveCharacterTextSplitter به قطعات کوچک تر تقسیم\nمی شوند. پارامترهایchunk_size   وchunk_overlap باید متناسب با زبان فارسی تنظیم شوند.\nمرحله۳: تبدیل به بردار و ذخیره سازی\nدر این مرحله، هر chunk   به یک بردار (Embedding) تبدیل شده و در پایگاه های داده برداری مانند\nChroma   یاFAISS ذخیره می شود.\nمرحله۴: ساخت زنجیره RAG\nدر نهایت، با استفاده ازRetrievalQA   یاLCEL (LangChain Expression Language)  ، زنجیره کامل\nبازیابی و تولید تعریف می شود.', parent_id='6'),
 ChildSearchResult(parent_title='معماری های پیشرفته RAG', child_title='Agentic R

In [12]:
from app.rag.decider import _format_chunks

a = _format_chunks(child_results)

In [13]:
print(a)

parent title :  اجزای اصلی یک سیستم   RAG
child title : مقدمه ای برای: اجزای اصلی یک سیستم   RAG
child content :
یک سیستم RAG   از چندین مؤلفه کلیدی تشکیل شده است که هر یک نقش مهمی در کیفیت نهایی
پاسخ ها ایفا می کنند.
parent id    :  2

---

parent title :  چالش های RAG برای زبان فارسی
child title : اتصال کلمات و توکن بندی
child content :
۳.۲  اتصال کلمات و توکن بندی
فارسی یک زبان پیوندی (Agglutinative)   است؛ به این معنا که پیشوندها و پسوندها به کلمات متصل
می شوند. برای مثال، واژه «کتاب هایم» از اجزای زیر تشکیل شده است:
•    کتاب
•   ها
•   ی
•   م
این ویژگی باعث می شود فرایند توکن بندی (Tokenization) در زبان فارسی پیچیده تر باشد و به
ابزارهای تخصصی نیاز داشته باشد. از جمله ابزارهای رایج برای این کار می توان به موارد زیر اشاره
کرد:
•   Hazm
•   NLTK Persian
استفاده از این ابزارها به تفکیک صحیح اجزای کلمات و بهبود کیفیت بازیابی اطالعات در سیستم های
RAG کمک می کند.
parent id    :  3

---

parent title :  اجزای اصلی یک سیستم   RAG
child title : پردازش و بارگذاری اسناد   (Document Ingesti

In [14]:
from sentence_transformers import CrossEncoder

reranker = CrossEncoder(
    "BAAI/bge-reranker-v2-m3"
)

Loading weights: 100%|██████████| 393/393 [00:00<00:00, 2530.84it/s]


In [1]:
from sentence_transformers import CrossEncoder


# بارگذاری مدل Re-ranker
reranker = CrossEncoder(
    "BAAI/bge-reranker-v2-m3"
)


def rerank_results(query, child_results, top_k=5):
    
    # ساخت زوج‌های (Query, Document)
    pairs = []

    for item in child_results:
        document = f"""
        موضوع اصلی:
        {item.parent_title}

        بخش:
        {item.child_title}

        محتوا:
        {item.child_content}
        """

        pairs.append(
            (query, document)
        )


    # گرفتن امتیاز ارتباط
    scores = reranker.predict(pairs)


    # اضافه کردن امتیاز به نتایج
    ranked_results = []

    for item, score in zip(child_results, scores):
        ranked_results.append(
            {
                # "child_id": item.child_id,
                "parent_id": item.parent_id,
                "parent_title": item.parent_title,
                "child_title": item.child_title,
                "child_content": item.child_content,
                "rerank_score": float(score)
            }
        )


    # مرتب سازی بر اساس امتیاز
    ranked_results = sorted(
        ranked_results,
        key=lambda x: x["rerank_score"],
        reverse=True
    )


    # فقط بهترین‌ها
    return ranked_results[:top_k]

d:\rag\venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
Loading weights: 100%|██████████| 393/393 [00:00<00:00, 6684.49it/s]


In [9]:
from app.services.retriever import search_children 



query = "اجزای rag را توضیح بده"

child_results = search_children(
    query,
    collection_name="loader",
    top_k=10
)


top_results = rerank_results(
    query,
    child_results,
    top_k=5
)


for r in top_results:
    print("------------------")
    print(r["child_title"])
    print(r["rerank_score"])

------------------
مقدمه ای برای: اجزای اصلی یک سیستم   RAG
0.9237738847732544
------------------
پردازش و بارگذاری اسناد   (Document Ingestion)
0.8839136958122253
------------------
بازیابی و رتبه بندی (Retrieval & Reranking)
0.4626826047897339
------------------
تقسیم بندی متن (Chunking)
0.45227521657943726
------------------
پایگاه داده برداری (Vector Store)
0.40097832679748535


In [10]:
child_results

[ChildSearchResult(parent_title='اجزای اصلی یک سیستم   RAG', child_title='مقدمه ای برای: اجزای اصلی یک سیستم   RAG', child_content='یک سیستم RAG   از چندین مؤلفه کلیدی تشکیل شده است که هر یک نقش مهمی در کیفیت نهایی\nپاسخ ها ایفا می کنند.', parent_id='2'),
 ChildSearchResult(parent_title='اجزای اصلی یک سیستم   RAG', child_title='پایگاه داده برداری (Vector Store)', child_content='۲.۴  پایگاه داده برداری (Vector Store)\nالگوریتم های جستجو در پایگاه های داده برداری معموالً بر اساس معیارهایی مانند شباهت کسینوسی\n(Cosine Similarity)   یا فاصله اقلیدسی (Euclidean Distance) عمل می کنند. برای مقیاس ،های بزرگ\nاز الگوریتم هایی نظیرHNSW (Hierarchical Navigable Small World)   وIVF (Inverted File Index)\nاستفاده می شود که جستجوی تقریبی نزدیک ترین همسایه ها (ANN - Approximate Nearest\nNeighbor) را با سرعت بسیار باال انجام می دهند.', parent_id='2'),
 ChildSearchResult(parent_title='اجزای اصلی یک سیستم   RAG', child_title='تقسیم بندی متن (Chunking)', child_content='۲.۲  تقسیم بندی متن (Chunking)\nپس ا

In [11]:
top_results

[{'parent_id': '2',
  'parent_title': 'اجزای اصلی یک سیستم   RAG',
  'child_title': 'مقدمه ای برای: اجزای اصلی یک سیستم   RAG',
  'child_content': 'یک سیستم RAG   از چندین مؤلفه کلیدی تشکیل شده است که هر یک نقش مهمی در کیفیت نهایی\nپاسخ ها ایفا می کنند.',
  'rerank_score': 0.9237738847732544},
 {'parent_id': '2',
  'parent_title': 'اجزای اصلی یک سیستم   RAG',
  'child_title': 'پردازش و بارگذاری اسناد   (Document Ingestion)',
  'child_content': '۲.۱  پردازش و بارگذاری اسناد   (Document Ingestion)\nاولین گام در ساخت یک سیستم RAG، آماده سازی و بارگذاری اسناد است. در این مرحله، متن خام از\nاسناد استخراج شده، پاک سازی می شود و برای مراحل بعدی آماده می گردد.\nاسناد می توانند در قالب های مختلفی مانند PDF   ، Word   ، HTML، متن ساده و حتی پایگاه های داده\nساخت یافته ارائه شوند. پس از استخراج محتوا، داده ها پردازش شده و برای استفاده در مراحل بعدی\nسیستم آماده می شوند.',
  'rerank_score': 0.8839136958122253},
 {'parent_id': '2',
  'parent_title': 'اجزای اصلی یک سیستم   RAG',
  'child_title': 'با

In [6]:
from newdecid import analyze_children

In [12]:
a = analyze_children(query, top_results, verbose=True)

[PARENT] child_title='مقدمه ای برای: اجزای اصلی یک سیستم   RAG' | reason=قطعه متن فقط به طور کلی اشاره می‌کند که سیستم RAG از چندین مؤلفه کلیدی تشکیل شده است و نقش آنها در کیفیت پاسخ‌ها مهم است، اما توضیح دقیقی درباره اجزای RAG ارائه نمی‌دهد. بنابراین برای پاسخ کامل به سوال باید کل متن والد بررسی شود.
[SKIP] child_title='پردازش و بارگذاری اسناد   (Document Ingestion)' -> parent قبلاً اضافه شده
[SKIP] child_title='بازیابی و رتبه بندی (Retrieval & Reranking)' -> parent قبلاً اضافه شده
[SKIP] child_title='تقسیم بندی متن (Chunking)' -> parent قبلاً اضافه شده
[SKIP] child_title='پایگاه داده برداری (Vector Store)' -> parent قبلاً اضافه شده


In [13]:
a

{'childs': [], 'parents': ['2']}

In [ ]:
pairs = []
for chunk in child_results:
    pairs.append((query, chunk["content"]))

scores = reranker.predict(pairs)

In [10]:
from collections import defaultdict

# child_results = خروجی Search

parents = defaultdict(list)

for child in child_results:
    parents[child.parent_id].append({
        "child_id": child.child_id,
        "child_title": child.child_title,
        "child_content": child.child_content,
    })

In [11]:
contexts = []

for parent_id, children in parents.items():

    parent_title = next(
        c.parent_title
        for c in child_results
        if c.parent_id == parent_id
    )

    text = f"Parent Title:\n{parent_title}\n\n"

    text += "Children:\n"

    for i, child in enumerate(children, start=1):
        text += (
            f"\n[{i}]\n"
            f"Child ID: {child['child_id']}\n"
            f"Title: {child['child_title']}\n"
            f"Content:\n{child['child_content']}\n"
        )

    contexts.append({
        "parent_id": parent_id,
        "context": text
    })

In [13]:
contexts[0]

{'parent_id': '2',
 'context': 'Parent Title:\nاجزای اصلی یک سیستم   RAG\n\nChildren:\n\n[1]\nChild ID: 2.1\nTitle: مقدمه ای برای: اجزای اصلی یک سیستم   RAG\nContent:\nیک سیستم RAG   از چندین مؤلفه کلیدی تشکیل شده است که هر یک نقش مهمی در کیفیت نهایی\nپاسخ ها ایفا می کنند.\n\n[2]\nChild ID: 2.2\nTitle: پردازش و بارگذاری اسناد   (Document Ingestion)\nContent:\n۲.۱  پردازش و بارگذاری اسناد   (Document Ingestion)\nاولین گام در ساخت یک سیستم RAG، آماده سازی و بارگذاری اسناد است. در این مرحله، متن خام از\nاسناد استخراج شده، پاک سازی می شود و برای مراحل بعدی آماده می گردد.\nاسناد می توانند در قالب های مختلفی مانند PDF   ، Word   ، HTML، متن ساده و حتی پایگاه های داده\nساخت یافته ارائه شوند. پس از استخراج محتوا، داده ها پردازش شده و برای استفاده در مراحل بعدی\nسیستم آماده می شوند.\n\n[3]\nChild ID: 2.7\nTitle: تولید پاسخ (Generation)\nContent:\n۲.۶    تولید پاسخ (Generation)\nدر مرحله نهایی، سؤال اصلی کاربر به همراه قطعات بازیابی شده به عنوانPrompt   به مدل زبانی بزرگ\nارسال می شود.\nساختار P